In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 3, Finished, Available, Finished, False)

In [2]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.dim.gold_product
(
    Product_ID       BIGINT,
    Product          STRING,
    Product_Category STRING,
    Created_TS       TIMESTAMP,
    Modified_TS      TIMESTAMP
)
USING DELTA;

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
Max_Date=spark.sql("select coalesce(max(Modified_TS),'1900-01-01') from LH_Silver.dim.gold_product").first()[0]

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 5, Finished, Available, Finished, False)

In [3]:
Max_Date

StatementMeta(, 3d973ab9-81e7-45e0-9cbc-ce903a15903a, 5, Finished, Available, Finished, False)

'1900-01-01'

In [4]:
df_bronze = spark.sql("""
    SELECT DISTINCT
        Product,
        Product_Category
    FROM Lakehouse_Bronze.dbo.Bronze_Sales
    WHERE Modified_TS > '{}'
      AND Product IS NOT NULL
""".format(Max_Date))

display(df_bronze)

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4660300a-9235-41cf-a863-ca485933a383)

In [5]:
Max_ID = spark.sql("""
    SELECT COALESCE(MAX(Product_ID), 0)
    FROM LH_Silver.dim.gold_product
""").first()[0]

print("Current Max Product_ID:", Max_ID)

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 7, Finished, Available, Finished, False)

Current Max Product_ID: 0


In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

window_spec = Window.orderBy("Product", "Product_Category")

df_final = (
    df_bronze
    .withColumn(
        "Product_ID",
        row_number().over(window_spec) + lit(Max_ID)
    )
    .withColumn("Created_TS", current_timestamp())
    .withColumn("Modified_TS", current_timestamp())
)

display(df_final)

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b66cbae8-0f74-46d4-91d0-1c1d98087afd)

In [7]:
df_final.createOrReplaceTempView("ViewProduct")

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 9, Finished, Available, Finished, False)

In [8]:
monotonically_increasing_id() + Max_ID + 1

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 10, Finished, Available, Finished, False)

Column<'((monotonically_increasing_id() + 0) + 1)'>

In [9]:
%%sql

MERGE INTO LH_Silver.dim.gold_product AS GP

USING ViewProduct AS VP

ON  GP.Product = VP.Product
AND GP.Product_Category = VP.Product_Category

WHEN MATCHED THEN

    UPDATE SET
        GP.Modified_TS = current_timestamp()

WHEN NOT MATCHED THEN

    INSERT
    (
        Product_ID,
        Product,
        Product_Category,
        Created_TS,
        Modified_TS
    )

    VALUES
    (
        VP.Product_ID,
        VP.Product,
        VP.Product_Category,
        current_timestamp(),
        current_timestamp()
    );

StatementMeta(, cf6186d2-07ad-4466-9060-da96ede06b61, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>